In [2]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, SVR
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from tqdm import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, \
RocCurveDisplay, roc_auc_score, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import log_loss, f1_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, BaggingClassifier, BaggingRegressor, RandomForestClassifier, RandomForestRegressor, StackingClassifier, StackingRegressor
from sklearn.linear_model import ridge_regression, ElasticNet
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, cross_val_score, KFold, StratifiedKFold

import xgboost as xgb
import lightgbm as lgb

import catboost

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
glass = pd.read_csv('https://raw.githubusercontent.com/dbda2025cdac-maker/Machine-Learning/refs/heads/main/Cases/Glass_Identification/Glass.csv')
x,y = glass.drop('Type', axis = 1), glass['Type']
le = LabelEncoder()
y = le.fit_transform(y)
lr = LogisticRegression()
kfold = KFold(n_splits=5, shuffle=True, random_state=25)

Grid search will work same as that of the nested loop that we were doing

In [5]:
params = {'solver' :['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
          'penalty' : ['l2', None],
           'C' : np.linspace(0.001, 15, 20) }
gcv = GridSearchCV(lr, param_grid=params, scoring='f1_macro', cv = kfold)
gcv.fit(x,y)

,estimator,LogisticRegression()
,param_grid,"{'C': array([1.0000...50000000e+01]), 'penalty': ['l2', None], 'solver': ['lbfgs', 'newton-cg', ...]}"
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,None


In [6]:
print(gcv.best_params_)
print(gcv.best_score_)

{'C': np.float64(0.001), 'penalty': None, 'solver': 'newton-cholesky'}
0.6223636233946888


In [8]:
df_cv = pd.DataFrame(gcv.cv_results_)
df_cv

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.030156,0.006123,0.003512,0.000616,0.001,l2,lbfgs,"{'C': 0.001, 'penalty': 'l2', 'solver': 'lbfgs'}",0.125281,0.077381,0.086207,0.081871,0.110812,0.096311,0.018539,198
1,0.010207,0.000947,0.003123,0.000222,0.001,l2,newton-cg,"{'C': 0.001, 'penalty': 'l2', 'solver': 'newto...",0.125281,0.077381,0.086207,0.081871,0.110812,0.096311,0.018539,198
2,0.006888,0.000673,0.003421,0.000957,0.001,l2,newton-cholesky,"{'C': 0.001, 'penalty': 'l2', 'solver': 'newto...",0.125281,0.077381,0.086207,0.081871,0.110812,0.096311,0.018539,198
3,0.003596,0.000278,0.002695,0.000353,0.001,l2,sag,"{'C': 0.001, 'penalty': 'l2', 'solver': 'sag'}",0.125281,0.077381,0.086207,0.081871,0.138740,0.101896,0.025110,196
4,0.005852,0.001074,0.003328,0.000810,0.001,l2,saga,"{'C': 0.001, 'penalty': 'l2', 'solver': 'saga'}",0.125281,0.077381,0.086207,0.081871,0.138740,0.101896,0.025110,196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,0.031857,0.000846,0.002869,0.000131,15.000,None,lbfgs,"{'C': 15.0, 'penalty': None, 'solver': 'lbfgs'}",0.512179,0.419658,0.454812,0.365684,0.551437,0.460754,0.065773,85
196,0.352195,0.096288,0.002992,0.000287,15.000,None,newton-cg,"{'C': 15.0, 'penalty': None, 'solver': 'newton...",0.561447,0.702514,0.625992,0.609711,0.557863,0.611505,0.052686,21
197,0.027883,0.009539,0.002888,0.000292,15.000,None,newton-cholesky,"{'C': 15.0, 'penalty': None, 'solver': 'newton...",0.646718,0.650847,0.591176,0.724294,0.498782,0.622364,0.074886,1
198,0.008177,0.000234,0.002501,0.000176,15.000,None,sag,"{'C': 15.0, 'penalty': None, 'solver': 'sag'}",0.284401,0.394100,0.411065,0.273810,0.414554,0.355586,0.062918,134


In [14]:
solvers = ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']
penalties = ['l2', None]
_c = np.linspace(0.001, 15, 20)
scores = []
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
for s in tqdm(solvers):
    for p in penalties:
        for _ in _c:
            try:
                lr = LogisticRegression(solver=s, penalty=p, max_iter=500, C=_)

                # using cross val score
                results = cross_val_score(lr, x,y, scoring='accuracy', cv = kfold)

                scores.append([_,s,p, np.mean(results)])
            except:
                pass
df_score = pd.DataFrame(scores, columns=['c', 'Solver', 'Penalty', 'Accuracy score'])
df_score.sort_values('Accuracy score', ascending=False)

100%|██████████| 5/5 [01:36<00:00, 19.22s/it]


,c,Solver,Penalty,Accuracy score
112,9.474053,newton-cholesky,None,0.635105
113,10.263474,newton-cholesky,None,0.635105
114,11.052895,newton-cholesky,None,0.635105
115,11.842316,newton-cholesky,None,0.635105
100,0.001000,newton-cholesky,None,0.635105
...,...,...,...,...
120,0.001000,sag,l2,0.331783
160,0.001000,saga,l2,0.331783
0,0.001000,lbfgs,l2,0.317497
40,0.001000,newton-cg,l2,0.317497


In [14]:
kfold = KFold(n_splits=5, shuffle=True, random_state=25)

params = {'max_depth' :[None, 3,4,5,6,7],
           'min_samples_split' :[2,10,0.025,0.01,0.05,0.1],
            'min_samples_leaf':[1,10,0.025,0.01,0.05,0.1],
            'random_state' :  [25]}

dtr = DecisionTreeClassifier()

gcv = GridSearchCV(dtr, param_grid=params, scoring = 'f1_macro', cv = kfold)
gcv.fit(x,y)

,estimator,DecisionTreeClassifier()
,param_grid,"{'max_depth': [None, 3, ...], 'min_samples_leaf': [1, 10, ...], 'min_samples_split': [2, 10, ...], 'random_state': [25]}"
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [15]:
print(gcv.best_params_)
print(gcv.best_score_)

{'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10, 'random_state': 25}
0.6746878572851875


In [13]:
depths = [None, 3,4,5,6,7]
min_samples = [2,10,0.025,0.01,0.05,0.1]
min_leaf = [1,10,0.025,0.01,0.05,0.1]
scores = []

kfold = KFold(n_splits=5, shuffle = True, random_state=25)

for d in tqdm(depths):
    for ms in min_samples:
        for ml in min_leaf:
            dtr = DecisionTreeClassifier(random_state=25, max_depth=d, min_samples_split=ms, min_samples_leaf=ml)
            # dtr.fit(x_train, y_train)
            results = cross_val_score(dtr, x,y,scoring = 'f1_macro')
            # y_pred = dtr.predict(x_test)
            scores.append([d,ms,ml,np.mean(results)])

df_score = pd.DataFrame(scores, columns=['depth', 'min_sample_split', 'min_sample_leaf', 'score'])
df_score.sort_values('score', ascending=False)

100%|██████████| 6/6 [00:06<00:00,  1.02s/it]


,depth,min_sample_split,min_sample_leaf,score
140,5.0,0.100,0.025,0.545244
32,NaN,0.100,0.025,0.543224
141,5.0,0.100,0.010,0.543217
138,5.0,0.100,1.000,0.542567
72,4.0,2.000,1.000,0.541371
...,...,...,...,...
59,3.0,0.010,0.100,0.424319
47,3.0,10.000,0.100,0.424319
41,3.0,2.000,0.100,0.424319
71,3.0,0.100,0.100,0.424319


In [ ]:
glass = pd.read_csv('https://raw.githubusercontent.com/dbda2025cdac-maker/Machine-Learning/refs/heads/main/Cases/Glass_Identification/Glass.csv')
x,y = glass.drop('Type', axis = 1), glass['Type']
le = LabelEncoder()
y = le.fit_transform(y)
lr = LogisticRegression()
kfold = KFold(n_splits=5, shuffle=True, random_state=25)
results = cross_val_score(lr, x,y, scoring='f1_macro', cv = kfold)
results